In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
logger = get_logger()

In [0]:
configs = parse_config_from_json("/Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/config/configs.json")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DecimalType, IntegerType, DateType
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

In [0]:
order_schema = StructType([StructField('row_id', IntegerType(), False),StructField('order_id', StringType(), False), StructField('order_date', DateType(), False), StructField('ship_date', DateType(), False), StructField('ship_mode', StringType(), False), StructField('customer_id', StringType(), False), StructField('product_id', StringType(), False), StructField('quantity', IntegerType(), False), StructField('price', DoubleType(), False), StructField('discount', DoubleType(), False), StructField('profit', DecimalType(10, 2), False)])

In [0]:
def parse_dates(df: DataFrame, date_formats: dict) -> DataFrame:
    """Parse date columns with specified formats."""
    for col, fmt in date_formats.items():
        df = df.withColumn(
            col,
            F.coalesce(
                F.to_date(F.col(col), fmt),
                F.to_date(F.col(col))
            )
        )
    return df

In [0]:
def fill_defaults(df: DataFrame, default_values: dict) -> DataFrame:
    """Fill missing values in a DataFrame with default values."""
    if df is None:
        raise ValueError("Cannot fill defaults for a None DataFrame.")
    if not default_values:
        return df

    try:
        logger.info("Filling missing values with defaults: %s", default_values)
        return df.fillna(default_values)
    except Exception as e:
        logger.error("Failed to fill defaults: %s", e)
        raise RuntimeError(f"Failed to fill defaults: {e}") from e

In [0]:
def dedupe(df: DataFrame, subset: list = None) -> DataFrame:
    """ Drop duplicate rows from a DataFrame."""
    
    return df.dropDuplicates(subset) if subset else df.dropDuplicates()

In [0]:
def apply_schema(df: DataFrame, schema: StructType) -> DataFrame:
    """Apply a schema to a DataFrame."""
    if df is None:
        raise ValueError("Cannot apply schema to a None DataFrame.")
    if not schema:
        raise ValueError("Schema must be a non-empty StructType.")
    try:
        result = df.select(*schema.names)
        logger.info("Schema applied: %s", schema.names)
        return result
    except Exception as e:
        logger.error("Failed to apply schema: %s", e)
        raise RuntimeError(f"Failed to apply schema: {e}") from e

In [0]:
def dq_orders(df: DataFrame) -> DataFrame:
    """
    Data quality checks for order records.
    Rules:
    - order_id: format XX-NNNN-NNNNNN 
    - customer_id: format XX-NNNNN
    - product_id: format XXX-XX-NNNNNNNN 
    - price: non-negative decimal 
    - discount: non-negative decimal 
    - profit: decimal value
    - quantity: positive integer
    """
    validated = df.filter(
        # order_id: not null/blank, matches XX-NNNN-NNNNNN pattern
        F.col("order_id").isNotNull() &
        (F.trim(F.col("order_id")) != "") &
        F.col("order_id").rlike(r"^[A-Z]{2}-\d{4}-\d{6}$") &

        # customer_id: not null/blank, matches XX-NNNNN pattern
        F.col("customer_id").isNotNull() &
        (F.trim(F.col("customer_id")) != "") &
        F.col("customer_id").rlike(r"^[A-Z]{2}-\d{5}$") &

        # product_id: not null/blank, matches XXX-XX-NNNNNNNN pattern
        F.col("product_id").isNotNull() &
        (F.trim(F.col("product_id")) != "") &
        F.col("product_id").rlike(r"^[A-Z]{3}-[A-Z]{2}-\d{8}$") &

        # price: not null, non-negative decimal
        F.col("price").isNotNull() &
        (F.col("price") >= 0) &

        # discount: not null, non-negative decimal
        F.col("discount").isNotNull() &
        (F.col("discount") >= 0) &

        # profit: not null (can be negative)
        F.col("profit").isNotNull() &

        # quantity: not null, positive integer
        F.col("quantity").isNotNull() &
        (F.col("quantity") > 0)
    )
    return validated


def build_fact_orders(df: DataFrame) -> DataFrame:
    """
    Build fact_orders from stg_orders:
    1. Check stg_orders table exists
    2. Apply defaults
    3. Validate records (dq_orders)
    4. Deduplicate by order_id
    5. Parse dates
    6. Apply schema
    7. Generate surrogate key (order_key)
    """

    defaults = configs["Orders"]['defaults']
    target_table = "az_ci_de_dbs.ecom_dev.fact_orders"

    # Apply defaults
    df = fill_defaults(df, defaults)

    # Data quality checks
    df = dq_orders(df)

    # Deduplicate by order_id
    df = dedupe(df)

    # Parse dates
    date_formats = {"order_date": "d/M/yyyy", "ship_date": "d/M/yyyy"}
    df = parse_dates(df, date_formats)

    # Apply schema
    df = apply_schema(df, order_schema)

    # Generate surrogate key
    df = df.withColumn("order_key", F.monotonically_increasing_id())

    delta_writer(df, target_table)

In [0]:
def main_build_orders():
    try:
        logger.info("Reading stg_orders table")
        df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_orders")
        logger.info("Building fact_orders")
        build_fact_orders(df)
        logger.info("fact_orders build completed successfully")
    except Exception as e:
        logger.error("Error in main_build_facts: %s", e)
        raise

In [0]:

def build_enriched_orders():
    """Join fact orders with dimension tables and return enriched DataFrame."""
    dim_products_df  = spark.read.table("az_ci_de_dbs.ecom_dev.dim_product")
    dim_customers_df = spark.read.table("az_ci_de_dbs.ecom_dev.dim_customer")
    fact_df   = spark.read.table("az_ci_de_dbs.ecom_dev.fact_orders")
    enriched = (
        fact_df.alias("o")
        .join(F.broadcast(dim_customers_df).alias("c"), on="customer_id", how="inner")
        .join(F.broadcast(dim_products_df).alias("p"),  on="product_id",  how="inner")
        .select(
            *[col for col in fact_df.columns if col not in ["customer_id", "product_id"]],
            "c.customer_name",
            "c.country",
            "p.category",
            "p.sub_category",
        )
    )
    return enriched

In [0]:
def main_build_enriched_orders():
    try:
        logger.info("Building enriched_orders DataFrame")
        enriched_df = build_enriched_orders()
        enriched_df = dedupe(enriched_df)
        target_table = "az_ci_de_dbs.ecom_dev.enriched_orders"
        delta_writer(enriched_df, target_table)
        logger.info("enriched_orders build completed successfully")
    except Exception as e:
        logger.error("Error in main_build_enriched_orders: %s", e)
        raise
